# 📚 Designing Data-Intensive Applications
## Bölüm 2: Defining Nonfunctional Requirements
### (İşlevsel Olmayan Gereksinimleri Tanımlamak)

---

> *"İnternet o kadar iyi yapıldı ki insanların çoğu onu, insan yapımı bir şey olarak değil, Pasifik Okyanusu gibi doğal bir kaynak olarak düşünüyor. Bu ölçekte bir teknolojinin en son ne zaman bu kadar hatasız olduğunu gördük?"*
> — Alan Kay, Dr. Dobb's Journal röportajı (2012)

---

Bir uygulama inşa ediyorsanız, bir gereksinimler listesine göre hareket edersiniz. Listenizin başında büyük olasılıkla **functional requirements** (işlevsel gereksinimler) yer alır: hangi ekranların ve düğmelerin olması gerektiği, her işlemin yazılımın amacını yerine getirmek için ne yapması gerektiği.

Bunlara ek olarak **nonfunctional requirements** (işlevsel olmayan gereksinimler) de vardır: uygulamanın hızlı, güvenilir, güvenli, yasal uyumlu ve bakımı kolay olması gibi. Bu gereksinimler açıkça yazıya dökülmemiş olabilir; çünkü bir ölçüde açık görünebilirler. Ancak en az uygulamanın işlevselliği kadar önemlidirler — dayanılmaz derecede yavaş ya da güvenilmez bir uygulama var olmayabileceği kadar değersizdir.

Bu bölümde şu konular ele alınacaktır:

| Konu | Açıklama |
|---|---|
| **Performance** | Bir sistemin performansını tanımlama ve ölçme |
| **Reliability** | Bir servisin güvenilir olmasının ne anlama geldiği — şeyler ters gittiğinde bile doğru çalışmaya devam etmek |
| **Scalability** | Sistem üzerindeki yük arttıkça computing kapasitesi eklemek için verimli yollar |
| **Maintainability** | Uzun vadede bir sistemi bakımının kolaylaştırılması |

---
## 🐦 Case Study: Social Network Home Timelines
### (Sosyal Ağ Ana Zaman Tüneli — Vaka Çalışması)

Soyut tanımlar oldukça kuru kalabilir. Fikirleri daha somut hale getirmek için bu bölüme bir sosyal ağ hizmetinin vaka çalışmasıyla başlanmaktadır.

X (eski adıyla Twitter) tarzında kullanıcıların mesaj gönderip diğer kullanıcıları takip edebildiği bir sosyal ağ implement etmekle görevlendirildiğimizi hayal edin.

### Varsayımlar

- Kullanıcılar günde toplam **500 milyon post** yapıyor → ortalama saniyede **5.800 post**
- Zirve anlarda bu oran **saniyede 150.000 post**'a kadar çıkabiliyor
- Ortalama bir kullanıcı **200 kişiyi** takip ediyor ve **200 takipçisi** var
- Ancak dağılım çok geniş: çoğu insanın yalnızca birkaç takipçisi varken Barack Obama gibi bazı ünlülerin 100 milyonun üzerinde takipçisi var

---
### 🗄️ Representing Users, Posts, and Follows
### (Kullanıcılar, Gönderiler ve Takip İlişkileri)

Tüm verileri bir **relational database**'de sakladığımızı varsayalım:
- Kullanıcılar için bir tablo (**users**)
- Gönderiler için bir tablo (**posts**)
- Takip ilişkileri için bir tablo (**follows**)

### Home Timeline Sorgusu

Sosyal ağımızın desteklemesi gereken ana okuma işlemi **home timeline**'dır — kullanıcının takip ettiği kişilerin son gönderilerini görüntüler. Belirli bir kullanıcının home timeline'ını almak için şu SQL sorgusu yazılabilir:

```sql
SELECT posts.*, users.*
FROM posts
  JOIN follows ON posts.sender_id = follows.followee_id
  JOIN users   ON posts.sender_id = users.id
WHERE follows.follower_id = current_user
ORDER BY posts.timestamp DESC
LIMIT 1000
```

Bu sorguyu çalıştırmak için database, `current_user`'ın takip ettiği herkesi bulmak üzere `follows` tablosunu kullanır, o kullanıcıların son gönderilerini arar ve en son 1.000 gönderiyi almak için `timestamp`'e göre sıralar.

### Sorun: 400 Milyon Lookup/Saniye

Gönderiler zamanında gösterilmeli; biri post yaptıktan sonra takipçilerinin bunu beş saniye içinde görebilmesini istiyoruz. Bir yaklaşım, kullanıcının clientının çevrimiçiyken bu sorguyu her beş saniyede bir tekrar etmesidir — buna **polling** denir.

Aynı anda 10 milyon kullanıcının çevrimiçi ve giriş yapmış olduğunu varsayarsak bu, saniyede **2 milyon** sorgu çalıştırmak anlamına gelir. Üstelik bu sorgu oldukça pahalıdır: bir kullanıcı 200 kişiyi takip ediyorsa sorgunun o 200 kişinin son gönderilerini getirmesi ve bu listeleri birleştirmesi gerekir.

**2 milyon timeline sorgusu/saniye × 200 takip edilen hesap = 400 milyon lookup/saniye** — çok büyük bir sayı. Ve bu ortalama durum; on binlerce hesabı takip eden bazı kullanıcılar için bu sorgu çalıştırılması çok pahalı ve hızlanması çok güçtür.

---
### 🗄️ Representing Users, Posts, and Follows
### (Kullanıcılar, Gönderiler ve Takip İlişkileri)

Tüm verileri bir **relational database**'de sakladığımızı varsayalım:
- Kullanıcılar için bir tablo (**users**)
- Gönderiler için bir tablo (**posts**)
- Takip ilişkileri için bir tablo (**follows**)

### Home Timeline Sorgusu

Sosyal ağımızın desteklemesi gereken ana okuma işlemi **home timeline**'dır — kullanıcının takip ettiği kişilerin son gönderilerini görüntüler. Belirli bir kullanıcının home timeline'ını almak için şu SQL sorgusu yazılabilir:

```sql
SELECT posts.*, users.*
FROM posts
  JOIN follows ON posts.sender_id = follows.followee_id
  JOIN users   ON posts.sender_id = users.id
WHERE follows.follower_id = current_user
ORDER BY posts.timestamp DESC
LIMIT 1000
```

Bu sorguyu çalıştırmak için database, `current_user`'ın takip ettiği herkesi bulmak üzere `follows` tablosunu kullanır, o kullanıcıların son gönderilerini arar ve en son 1.000 gönderiyi almak için `timestamp`'e göre sıralar.

### Sorun: 400 Milyon Lookup/Saniye

Gönderiler zamanında gösterilmeli; biri post yaptıktan sonra takipçilerinin bunu beş saniye içinde görebilmesini istiyoruz. Bir yaklaşım, kullanıcının clientının çevrimiçiyken bu sorguyu her beş saniyede bir tekrar etmesidir — buna **polling** denir.

Aynı anda 10 milyon kullanıcının çevrimiçi ve giriş yapmış olduğunu varsayarsak bu, saniyede **2 milyon** sorgu çalıştırmak anlamına gelir. Üstelik bu sorgu oldukça pahalıdır: bir kullanıcı 200 kişiyi takip ediyorsa sorgunun o 200 kişinin son gönderilerini getirmesi ve bu listeleri birleştirmesi gerekir.

**2 milyon timeline sorgusu/saniye × 200 takip edilen hesap = 400 milyon lookup/saniye** — çok büyük bir sayı. Ve bu ortalama durum; on binlerce hesabı takip eden bazı kullanıcılar için bu sorgu çalıştırılması çok pahalı ve hızlanması çok güçtür.

---
### ⚡ Materializing and Updating Timelines
### (Timeline'ları Önceden Hesaplama ve Güncelleme)

Daha iyi nasıl yapabiliriz?

1. **Polling yerine push**: Server, çevrimiçi olan takipçilere yeni gönderileri aktif olarak iletmeli.
2. **Precompute (önceden hesaplama)**: Sorgunun sonuçlarını önceden hesaplamalı; böylece kullanıcının home timeline isteği bir **cache**'den karşılanabilir.

### Yaklaşım: Her Kullanıcı İçin Timeline Cache

Her kullanıcı için home timeline'ını (yani takip ettiği kişilerin son gönderilerini) içeren bir veri yapısı saklayalım. Bir kullanıcı post yaptığında, tüm takipçilerini bulur ve o gönderiyi her takipçinin home timeline'ına ekleriz — bir posta kutusuna mektup teslim etmek gibi.

Artık bir kullanıcı giriş yaptığında, bu önceden hesaplanmış home timeline'ı onlara sunabiliriz. Üstelik yeni gönderiler hakkında bildirim almak için kullanıcının clientının home timeline'larına eklenen gönderilerin akışına abone olması yeterlidir.

### Fan-Out

Bu yaklaşımın dezavantajı, home timeline'lar güncellenmesi gereken **derived data** (türetilmiş veri) olduğu için bir kullanıcı post yaptığında artık daha fazla iş yapmamız gerektiğidir.

Bir başlangıç isteği birden fazla downstream isteğinin gerçekleştirilmesiyle sonuçlandığında, istek sayısının kaç katına çıktığını tanımlamak için **fan-out** terimi kullanılır.

**Hesap:**
- Saniyede 5.800 post × 200 takipçi (fan-out faktörü = 200)
- = Saniyede **~1 milyon** home timeline yazma işlemi

Bu çok fazladır; ama 400 milyon lookup/saniye'ye kıyasla **önemli bir tasarruf**.

### Yük Artışlarında Davranış

Özel bir etkinlik nedeniyle post oranı artarsa, timeline teslimlerini hemen yapmak zorunda değiliz — bunları **queue**'ya (kuyruğa) alıp takipçilerin timeline'larında görünmesinin geçici olarak biraz daha uzun süreceğini kabul edebiliriz. Bu tür yük artışları sırasında bile timeline'lar hızlı yüklenir, çünkü yalnızca bir cache'den servis ederiz.

### Materialization (Somutlaştırma)

Bir sorgunun sonuçlarını önceden hesaplama ve güncelleme sürecine **materialization** denir; timeline cache ise bir **materialized view**'ın (somutlaştırılmış görünüm) örneğidir. Materialized view, okumaları hızlandırır; ancak karşılığında yazmalarda daha fazla iş yapılması gerekir.

### Köşe Durumlar: Extreme Cases

Yazma maliyeti çoğu kullanıcı için mütevazıdır; ancak sosyal ağın bazı aşırı durumları da göz önünde bulundurması gerekir:

1. **Çok fazla hesap takip eden kullanıcı**: Eğer bir kullanıcı çok sayıda hesabı takip ediyorsa ve bu hesaplar çok post yapıyorsa, bu kullanıcının materialized timeline'ına yüksek oranda yazma yapılacaktır. Ancak bu kullanıcı muhtemelen timeline'ındaki tüm gönderileri okumuyordur; bu nedenle bazı timeline yazmalarını silip kullanıcıya takip ettiği hesapların gönderilerinden yalnızca bir örnek göstermek kabul edilebilir.

2. **Ünlü hesaplar (celebrity problem)**: Milyonlarca takipçisi olan ünlü bir hesap post yaptığında, o gönderiyi milyonlarca takipçinin home timeline'ına eklemek büyük iş yükü oluşturur. Bu durumda bazı yazmaları silmek kabul edilemez. Bir çözüm yolu, ünlü gönderilerini diğerlerinden farklı ele almaktır: ünlü gönderilerini milyonlarca timeline'a ekleme zahmetinden kurtulmak için bunları ayrı saklayıp materialized timeline okunduğunda birleştirilebiliriz.

---
## 📊 Performance Nasıl Tanımlanır?
## Describing Performance

Yazılım performansının çoğu tartışması iki ana metrik türünü ele alır:

### Response Time (Yanıt Süresi)
Bir kullanıcının istek yaptığı andan istenen yanıtı alana kadar geçen süre. Ölçüm birimi saniyedir (veya milisaniye, mikrosaniye).

### Throughput (Verim)
Sistemin işlediği saniyedeki istek sayısı veya saniyedeki veri hacmi. Belirli bir donanım kaynakları tahsisi için işlenebilecek maksimum bir throughput vardır. Ölçüm birimi "saniyedeki bir şeyler" şeklindedir.

Sosyal ağ vaka çalışmasında:
- **"Saniyedeki post sayısı"** ve **"saniyedeki timeline yazması"** → throughput metrikleri
- **"Home timeline yükleme süresi"** ve **"bir postun takipçilere ulaşma süresi"** → response time metrikleri

### Throughput ve Response Time İlişkisi

Throughput ve response time genellikle ilişkilidir. Online servisler için bu ilişki şu şekildedir:
- İstek throughput'u düşükken servisin **response time** düşüktür
- Yük arttıkça **response time** artmaya başlar

Bunun nedeni **queueing** (kuyruk beklemesi)dir: yoğun yüklü bir sistemde bir istek geldiğinde CPU muhtemelen daha önceki bir isteği işlemektedir, dolayısıyla gelen istek önceki tamamlanana kadar beklemek zorundadır. Throughput donanımın kaldırabileceği maksimuma yaklaştıkça **queueing delay**'ler keskin biçimde artar.

---
### 🌀 Metastable Failure: Aşırı Yüklü Bir Sistem Toparlanamadığında
### When an Overloaded System Won't Recover

Bir sistem aşırı yüke yakınsa — throughput limite yaklaşmışsa — bazen kısır bir döngüye girebilir: daha az verimli hale gelir ve bu da onu daha da fazla aşırı yüklü kılar.

Örneğin, uzun bir istek kuyruğu bekliyor olduğunda response time'lar o kadar artabilir ki client'lar **timeout** yapar ve isteklerini yeniden gönderir. Bu da istek oranını daha da artırarak durumu kötüleştirir — **retry storm** (yeniden deneme fırtınası). Yük azaltılsa bile böyle bir sistem yeniden başlatılana veya başka şekilde sıfırlanana kadar aşırı yüklü durumda kalabilir. Bu olguya **metastable failure** denir; production sistemlerinde ciddi kesintilere yol açabilir.

### Metastable Failure'ı Önleme Yöntemleri

| Yöntem | Açıklama |
|---|---|
| **Exponential backoff** | Client tarafında ardışık yeniden denemeler arasındaki süreyi artırma ve rastgele hale getirme |
| **Circuit breaker** | Son zamanlarda hata döndüren veya timeout olan bir servise istek göndermeyi geçici olarak durdurma |
| **Token bucket algorithm** | İstek hızını kontrol etmek için token tabanlı algoritma |
| **Load shedding** | Sunucunun aşırı yüke yaklaştığını tespit ettiğinde istekleri proaktif olarak reddetmeye başlaması |
| **Backpressure** | Sunucunun client'lara yavaşlamalarını isteyen yanıtlar göndermesi |

### Performance Metriklerinde Öncelik

- **Response time** → kullanıcıların en çok önem verdiği şey
- **Throughput** → gerekli computing kaynaklarını (kaç sunucu gerektiği gibi) ve dolayısıyla belirli bir iş yükünü sunmanın maliyetini belirler

Throughput'un mevcut donanımın kapasitesini aşması muhtemel görünüyorsa kapasitenin genişletilmesi gerekir. Computing kaynakları eklenerek maksimum throughput'u önemli ölçüde artırılabilen bir sistem **scalable** (ölçeklenebilir) olarak adlandırılır.

---
### ⏱️ Latency ve Response Time Farkı
### Latency and Response Time

"Latency" ve "response time" zaman zaman birbirinin yerine kullanılır; ancak bu kitapta bu ve birkaç ilgili terim belirli şekillerde kullanılır:

| Terim | Tanım |
|---|---|
| **Response time** | Client'ın gördüğü süre; sistemin herhangi bir yerinde oluşan tüm gecikmeleri kapsar |
| **Service time** | Servisin client isteğini aktif olarak işlediği süre |
| **Queueing delay** | Akıştaki çeşitli noktalarda oluşan bekleme süresi (örn. CPU uygun olana kadar bekleme, ağ tamponu) |
| **Latency** | Bir isteğin aktif olarak işlenmediği süre için genel terim; özellikle **network latency** isteğin ve yanıtın ağda seyahat ettiği süreyi ifade eder |

### Response Time Neden Değişkendir?

Response time, aynı isteği tekrar tekrar yapsanız bile istekten isteğe önemli ölçüde değişebilir. Birçok faktör rastgele gecikmeler ekleyebilir:
- Bir background işlemine **context switch**
- Network paketi kaybı ve **TCP retransmission**
- **Garbage collection** duraklaması
- Diskten okumayı zorlamak için **page fault**
- Sunucu rafındaki mekanik titreşimler

### Head-of-Line Blocking

Queueing gecikmeleri response time değişkenliğinin büyük bölümünü oluşturur. Bir sunucu paralel olarak yalnızca az sayıda şeyi işleyebildiğinden (örneğin CPU çekirdeği sayısıyla sınırlı), sonraki isteklerin işlenmesini engellemek için yalnızca az sayıda yavaş istek yeterlidir — buna **head-of-line blocking** denir.

Sonraki istekler hızlı service time'a sahip olsa bile, önceki isteğin tamamlanmasını beklemek için geçen süre nedeniyle client yavaş bir genel response time görür. Queueing delay, service time'ın bir parçası değildir; bu nedenle response time'ları **client tarafında ölçmek** önemlidir.

---
### 📈 Percentile'lar: Ortalama, Medyan ve Yüzdelikler
### Average, Median, and Percentiles

Response time istekten isteğe değiştiğinden, bunu tek bir sayı olarak değil, ölçebileceğimiz **değer dağılımı** olarak düşünmeliyiz.

### Ortalama (Mean) Neden Yetersizdir?

Bir servisin ortalama response time'ını raporlamak yaygındır (tüm response time'ları toplayıp istek sayısına bölerek elde edilen **aritmetik ortalama**). Ortalama response time throughput limitlerini tahmin etmek için kullanışlıdır. Ancak "tipik" response time'ı öğrenmek istiyorsanız ortalama iyi bir metrik değildir — çünkü kaç kullanıcının aslında o gecikmeyi yaşadığını söylemez.

### Percentile Nedir?

Response time listenizi en hızlıdan en yavaşa sıralarsanız, **median** (medyan) orta noktadır:
- Median response time 200 ms ise → isteklerinizin yarısı 200 ms'den daha kısa sürede geri döner, yarısı daha uzun sürer
- Medyan, kullanıcıların ne kadar beklemesi gerektiğini öğrenmek istediğinizde iyi bir metriktir
- Medyan aynı zamanda **50th percentile** veya **p50** olarak da bilinir

### Yüksek Percentile'lar ve Tail Latency

Aykırı değerlerinizin ne kadar kötü olduğunu anlamak için daha yüksek percentile'lara bakabilirsiniz:

| Percentile | Anlamı |
|---|---|
| **p50** (median) | 100 istekten 50'si bu sürenin altında tamamlanır |
| **p95** | 100 istekten 95'i bu sürenin altında; 5'i bu süreyi aşar |
| **p99** | 100 istekten 99'u bu sürenin altında; 1'i bu süreyi aşar |
| **p999** | 1.000 istekten 999'u bu sürenin altında; 1'i bu süreyi aşar |

Yüksek response time percentile'larına, aynı zamanda **tail latencies** olarak da bilinir; bunlar kullanıcıların servis deneyimini doğrudan etkiler.

**Amazon'dan Örnek:** Amazon, dahili servisler için response time gereksinimlerini **99.9th percentile** cinsinden tanımlar — bu yalnızca 1.000 istekten 1'ini etkiliyor olsa da. Bunun nedeni, en yavaş istekleri olan müşterilerin genellikle hesaplarında en fazla verisi olan müşteriler — yani en değerli müşteriler olmasıdır.

**99.99th percentile**'ı optimize etmek ise çok pahalı bulunmuş ve Amazon için yeterince fayda sağlamadığı görülmüştür. Çok yüksek percentile'larda response time'ları azaltmak güçtür; çünkü bunlar kontrolünüz dışındaki rastgele olaylardan kolayca etkilenir ve faydalar azalmaktadır.

### Kullanıcı Etkisi Üzerine Araştırmalar

Hızlı bir servisin yavaş bir servisten kullanıcılar için daha iyi olduğu açık görünür, ancak gecikmenin kullanıcı davranışı üzerindeki etkisini ölçmek şaşırtıcı ölçüde güçtür:

- 2006'da Google, arama sonuçlarındaki 400 ms'den 900 ms'ye yavaşlamanın trafik ve gelirde %20 düşüşle ilişkili olduğunu bildirdi.
- 2009'da başka bir Google araştırması, 400 ms gecikme artışının yalnızca günlük %0.6 daha az aramaya yol açtığını ortaya koydu.
- Aynı yıl Bing, 2 saniyelik yükleme süresi artışının reklam gelirini %4.3 azalttığını keşfetti.
- Bir Yahoo araştırması, hızlı ve yavaş yüklenen arama sonuçları arasındaki fark 1.25 saniye veya daha fazla olduğunda hızlı aramalarda %20–30 daha fazla tıklama olduğunu bildiriyor.

Bu çalışmaların sonuçları metodolojik sorunlar nedeniyle tartışmalı olmaya devam etmektedir; dikkatli yorumlanmalıdır.

---
### 🔗 Tail Latency Amplification ve Percentile Kullanımı
### Use of Response Time Metrics

### Tail Latency Amplification

Yüksek percentile'lar, tek bir son kullanıcı isteğine hizmet etmek için birden fazla kez çağrılan **backend servislerinde** özellikle önemlidir.

Çağrıları paralel yapsanız bile istek yine de en yavaş paralel çağrının tamamlanmasını beklemek zorundadır. Yalnızca bir yavaş çağrı tüm son kullanıcı isteğini yavaşlatmak için yeterlidir.

Backend çağrılarının yalnızca küçük bir yüzdesi yavaş olsa bile, bir son kullanıcı isteği birden fazla backend çağrısı gerektiriyorsa yavaş çağrıya denk gelme olasılığı artar; bu nedenle son kullanıcı isteklerinin daha büyük bir oranı yavaş olmaya mahkûm olur. Bu etkiye **tail latency amplification** (kuyruk gecikmesi amplifikasyonu) denir.

### SLO ve SLA

Percentile'lar genellikle bir servisin beklenen performansını ve kullanılabilirliğini tanımlamanın yolları olarak **SLO (Service Level Objective)** ve **SLA (Service Level Agreement)**'larda kullanılır:

| Terim | Tanım |
|---|---|
| **SLO** | Dahili hedef: Örneğin median response time 200 ms'den az, p99 1 saniyenin altında, geçerli isteklerin en az %99.9'u hata almadan sonuçlanır |
| **SLA** | SLO karşılanmazsa ne olacağını belirten bir sözleşme (örn. müşteriler geri ödeme almaya hak kazanabilir) |

Pratikte SLO ve SLA için iyi availability metrikleri tanımlamak basit değildir.

### Percentile Hesaplama

Hizmetleriniz için izleme dashboard'larına response time percentile'ları eklemek istiyorsanız, bunları sürekli olarak verimli şekilde hesaplamanız gerekir. Örneğin son 10 dakikadaki istekler için rolling window (kayan pencere) tutup her dakika o penceredeki değerler üzerinden medyanı ve çeşitli percentile'ları hesaplayabilirsiniz.

En basit uygulama, tüm isteklerin response time'larını listede tutmak ve listeyi her dakika sıralamaktır. Bu çok verimsizse, minimum CPU ve bellek maliyetiyle percentile'ların iyi bir yaklaşımını hesaplayabilen algoritmalar mevcuttur:

- **HdrHistogram**: Yüksek dinamik aralıklı histogram kütüphanesi
- **t-digest**: Dağıtımlarda verimli tahminler yapan algoritma
- **OpenHistogram**: Açık kaynaklı histogram kütüphanesi
- **DDSketch**: Hızlı ve tamamen birleştirilebilir quantile tahmini

> ⚠️ **Uyarı**: Percentile'ları ortalama almak (zaman çözünürlüğünü azaltmak veya birkaç makineden veri birleştirmek için) matematiksel olarak anlamsızdır. Response time verilerini bir araya getirmenin doğru yolu **histogram'ları toplamaktır**.

---
## 🛡️ Reliability and Fault Tolerance
### (Güvenilirlik ve Hata Toleransı)

Herkesin bir şeyin güvenilir ya da güvenilmez olmasının ne anlama geldiğine dair sezgisel bir fikri vardır. Yazılım için tipik beklentiler şunlardır:

- Uygulama, kullanıcının beklediği işlevi yerine getirir
- Uygulama, kullanıcının hata yapmasına veya yazılımı beklenmedik şekillerde kullanmasına tolerans gösterebilir
- Performansı, gerekli yük ve veri hacmi altında beklenen kullanım senaryosu için yeterince iyidir
- Sistem yetkisiz erişimi ve kötüye kullanımı engeller

Tüm bu şeyler birlikte "doğru çalışmak" anlamına geliyorsa, **reliability**'yi (güvenilirlik) kabaca "şeyler ters gittiğinde bile doğru çalışmaya devam etmek" olarak anlayabiliriz.

### Fault vs Failure Ayrımı

| Terim | Tanım |
|---|---|
| **Fault** (Arıza) | Sistemin belirli bir parçasının doğru çalışmayı durdurması — örneğin tek bir hard diskin arızalanması, tek bir makinenin çökmesi veya bir dış servisin kesintiye uğraması |
| **Failure** (Başarısızlık) | Sistemin bir bütün olarak kullanıcıya gerekli hizmeti sağlamayı durdurması — başka bir deyişle SLO'yu karşılamaması |

**Fark neden önemlidir?**

Bir hard disk çalışmayı durursa, hard diskin arızalandığını söyleriz (**fault**). Sistem yalnızca o tek hard diskten oluşuyorsa, gerekli hizmet sağlamayı durdurdu ve dolayısıyla o da başarısız olmuştur (**failure**). Ancak sistem birden fazla hard diskten oluşuyorsa, tek bir hard diskin arızası daha büyük sistemin bakış açısından yalnızca bir **fault**'tur ve daha büyük sistem, başka bir hard diskte verinin kopyası olduğu için bu hataya tolerans gösterebilir.

---
### 🔧 Fault Tolerance ve SPOF

Belirli fault'ların oluşmasına rağmen kullanıcılara gerekli hizmeti sağlamaya devam eden bir sisteme **fault-tolerant** (hata toleranslı) deriz.

Belirli bir parçanın arızalanmasına tolerans gösteremeyen bir sistem için bu kısma **SPOF (Single Point of Failure)** deriz; çünkü o kısımdaki bir fault, tüm sistemin başarısızlığına yol açan bir fault'a dönüşür.

### Sosyal Ağ Örneği

Sosyal ağ vaka çalışmasında olası bir fault şudur: fan-out süreci sırasında, materialized timeline'ları güncelleyen bir makine çöker ya da erişilemez hale gelir. Bu süreci fault-tolerant yapmak için başka bir makinenin, teslim edilmesi gereken hiçbir post'u kaçırmadan ve herhangi bir post'u yinelemeden bu görevi devralabilmesini sağlamamız gerekir. Bu fikir **exactly-once semantics** (tam olarak bir kez semantiği) olarak bilinir.

### Fault Toleransın Sınırları

Fault toleransı her zaman belirli türde ve sayıda fault ile sınırlıdır. Örneğin bir sistem, aynı anda en fazla iki hard diskin arızalanmasına veya üç node'dan birinin çökmesine tolerans gösterebilir. Herhangi sayıda fault'u tolere etmek mantıklı değildir; tüm node'lar çöküyorsa yapılabilecek hiçbir şey yoktur.

### Fault Injection ve Chaos Engineering

Sezgiye aykırı biçimde, bu tür fault-tolerant sistemlerde fault oranını artırarak kasıtlı olarak tetiklemek mantıklı olabilir — örneğin bireysel süreçleri uyarı vermeden rastgele sonlandırarak. Buna **fault injection** (hata enjeksiyonu) denir.

Kritik hataların çoğu aslında zayıf hata yönetiminden kaynaklanmaktadır. Fault'ları kasıtlı olarak tetikleyerek, fault-tolerance mekanizmasının sürekli çalıştırıldığından ve test edildiğinden emin olursunuz; bu da fault'ların doğal olarak oluştuğunda doğru şekilde ele alınacağına güveninizi artırabilir.

**Chaos engineering**, kasıtlı olarak fault enjekte etmek gibi deneyler aracılığıyla fault-tolerance mekanizmalarına güveni artırmayı amaçlayan bir disiplindir.

Genel olarak fault'ları önlemeye karşı tolere etmeyi tercih etsek de bazen önlemenin tedaviden daha iyi olduğu durumlar vardır (örneğin hiçbir çözüm yoksa). Güvenlik meseleleri buna örnek verilebilir: bir saldırgan sistemi tehlikeye atıp hassas verilere erişmişse, bu olay geri alınamaz.

---
### 💾 Hardware ve Software Faults
### Hardware and Software Faults

Sistem arızasının nedenlerini düşündüğümüzde, akla ilk **hardware faults** (donanım arızaları) gelir:

### Donanım Arıza İstatistikleri

| Bileşen | Arıza Oranı / Davranış |
|---|---|
| **Manyetik hard disk** | Yılda yaklaşık %2–5 arıza; 10.000 diskli bir depolama kümesinde günde ortalama bir disk arızası beklenir |
| **SSD** | Yılda yaklaşık %0.5–1 arıza; küçük bit hataları otomatik düzeltilir, ancak düzeltilemez hatalar oldukça yeni disklerde bile yılda yaklaşık bir kez olur — bu hata oranı manyetik hard disklerden yüksektir |
| **CPU** | Yaklaşık 1.000 makinede birinde, muhtemelen üretim hatası nedeniyle zaman zaman yanlış sonuç hesaplayan bir CPU çekirdeği bulunur; bazı durumlarda yanlış hesaplama çökmeye yol açar, bazılarında ise program yanlış sonuç döndürür |
| **RAM** | Kozmik ışınlar veya kalıcı fiziksel kusurlar nedeniyle bozulabilir; **ECC (Error-Correcting Code)** bellekle bile %1'den fazla makine, genellikle makinenin çökmesine yol açan düzeltilemez bir hatayla karşılaşır |
| **Tüm datacenter** | Güç kesintisi, network yanlış yapılandırması veya kalıcı hasar (yangın, sel, deprem) nedeniyle erişilemez hale gelebilir |
| **Güneş fırtınası** | Uzun mesafeli tellerde büyük elektrik akımları oluşturarak elektrik şebekelerine ve denizaltı ağ kablolarına zarar verebilir |

Bu olaylar küçük bir sistemde sizi nadiren endişelendirmeye değer; ancak büyük ölçekli bir sistemde donanım arızaları normal sistem operasyonunun bir parçası haline gelir.

### Redundancy ile Donanım Arızalarına Tolerans Gösterme

Güvenilmez donanıma ilk yanıt genellikle sistemin arıza oranını azaltmak için bireysel donanım bileşenlerine **redundancy** (yedeklilik) eklemektir:

- **RAID** yapılandırmasında diskler (aynı makinede birden fazla disk arasında veri yayarak diskten kaynaklanan veri kaybının önlenmesi)
- Çift güç kaynağına sahip sunucular ve hot-swappable (çalışır haldeyken değiştirilebilir) CPU'lar
- Yedek güç için batarya ve dizel generatörlere sahip datacenter'lar

Redundancy, bileşen arızaları **bağımsız** olduğunda — yani bir arızanın başka bir arızanın olasılığını değiştirmediği durumlarda — en etkilidir. Ancak deneyim, bileşen arızaları arasında önemli korelasyonlar bulunduğunu göstermektedir.

Bu nedenle cloud sistemleri, bireysel makinelerin güvenilirliğine daha az odaklanma ve bunun yerine yazılım düzeyinde hatalı node'lara tolerans göstererek servisleri yüksek oranda erişilebilir kılma eğilimindedir. Cloud provider'lar, hangi kaynakların fiziksel olarak aynı konumda olduğunu belirlemek için **availability zone**'ları kullanır.

### Rolling Upgrade

Tek node'ları tolere edebilen sistemlerin operasyonel avantajları da vardır. Tek sunuculu bir sistem, makineyi yeniden başlatmanız gerekiyorsa (örneğin işletim sistemi güvenlik yamaları uygulamak için) planlanmış downtime gerektirir. Oysa çok node'lu fault-tolerant bir sistem, kullanıcılara yönelik hizmeti etkilemeden her seferinde bir node'u yeniden başlatarak yamalanabilir. Buna **rolling upgrade** (kademeli güncelleme) denir.

---
### 🐛 Software Faults (Yazılım Arızaları)

Donanım arızaları zayıf şekilde ilişkili olsa da büyük ölçüde bağımsızdır. Buna karşın **software faults** (yazılım arızaları) çoğu zaman çok yüksek düzeyde ilişkilidir; çünkü pek çok node'un aynı yazılımı çalıştırması ve dolayısıyla aynı bug'lara sahip olması yaygındır.

Bu tür arızaların öngörülmesi daha güçtür ve ilişkisiz donanım arızalarından çok daha fazla sistem hatasına yol açma eğilimindedir.

### Software Fault Örnekleri

1. **Aynı anda tüm node'ların arızalanmasına yol açan software bug:**
   - 30 Haziran 2012'de bir artık saniye, Linux kernel'deki bir bug nedeniyle pek çok Java uygulamasının aynı anda askıya girmesine ve birkaç internet servisinin çökmesine yol açtı.
   - Bir firmware bug'ı nedeniyle belirli modellerdeki tüm SSD'ler tam olarak **32.768 saat** çalıştıktan sonra birdenbire arızalanır (dört yıldan az), üzerindeki verileri kurtarılamaz hale getirir.

2. **Runaway process** (Kontrolden çıkmış süreç): CPU zamanı, bellek, disk alanı, network bant genişliği veya thread'ler gibi paylaşılan, sınırlı bir kaynağı tüketen süreç.

3. **Bağımlı servis sorunları**: Sistemin bağımlı olduğu bir servisin yavaşlaması, yanıt vermemesi veya bozuk yanıtlar döndürmeye başlaması.

4. **Emergent behavior** (Ortaya çıkan davranış): Farklı sistemler arasındaki etkileşim, her sistem ayrı ayrı test edildiğinde görülmeyen beklenmedik davranışlara yol açar.

5. **Cascading failures** (Zincirleme arızalar): Bir bileşendeki sorunun başka bir bileşenin aşırı yüklenmesine ve yavaşlamasına neden olması, bu durumun da başka bir bileşeni çökertmesi.

### Software Fault'larla Başa Çıkma

Software'deki sistematik arızalar için hızlı bir çözüm yoktur. Pek çok küçük şey yardımcı olabilir:
- Sistemdeki varsayımlar ve etkileşimler üzerine dikkatli düşünmek
- Kapsamlı test
- **Process isolation** sağlamak
- Süreçlerin çöküp yeniden başlamasına izin vermek
- **Retry storm** gibi feedback döngülerinden kaçınmak
- Production'da sistem davranışını ölçmek, izlemek ve analiz etmek

---
### 👤 İnsan Hatası ve Güvenilirlik
### Humans and Reliability

İnsanlar yazılım sistemlerini tasarlar ve inşa eder; sistemlerin çalışmaya devam etmesini sağlayan operatörler de insandır. Makinelerin aksine insanlar kurallara uymaz; güçlü yönlerinden biri, işlerini yaparken yaratıcı ve uyumlu olmalarıdır. Ancak bu özellik aynı zamanda öngörülemezliğe ve en iyi niyetlere rağmen hatalara yol açar.

**Önemli bir araştırma:** Büyük internet servislerini inceleyen bir çalışma, operatörlerin konfigürasyon değişikliklerinin kesintilerin en önemli nedeni olduğunu ortaya koydu; donanım arızaları (sunucu veya ağ) yalnızca %10–25'lik olaylarda rol oynadı.

### Suçlama Kültürü Yerine Sistemsel Düşünme

Bu tür sorunları **"insan hatası"** olarak etiketlemek ve tighter prosedürler ile kurallara uyum aracılığıyla insan davranışını daha iyi kontrol ederek çözülebileceğini ummak cazip gelir. Ancak insanları hatalar için suçlamak üretken değildir.

"İnsan hatası" dediğimiz şey aslında bir olayın nedeni değil, insanların işlerini yapmaya çalıştığı **sosyoteknik sistemdeki** bir sorunun belirtisidir. Genellikle karmaşık sistemler, bileşenler arasındaki beklenmedik etkileşimlerin de arızalara yol açabildiği emergent davranışa sahiptir.

### İnsan Hatasının Etkisini Azaltmak

Çeşitli teknik önlemler insan hatalarının etkisini en aza indirmeye yardımcı olabilir:

- Kapsamlı test (hem elle yazılmış testler hem de çok sayıda rastgele girdide **property testing**)
- Konfigürasyon değişikliklerini hızlıca geri almak için **rollback mekanizmaları**
- Yeni kodun **gradual rollout**'u (kademeli yayılımı)
- Ayrıntılı ve net izleme
- Production sorunlarını teşhis etmek için **observability tools**
- "Doğru şeyi" teşvik eden ve "yanlış şeyi" caydıran iyi tasarlanmış arayüzler

Ancak bunların tümü zaman ve para yatırımı gerektirir. Pragmatik iş gerçekliğinde organizasyonlar genellikle sistemlerinin hatalara karşı dayanıklılığını artıracak önlemler yerine gelir getiren faaliyetlere öncelik verir. Daha fazla özellik ile daha fazla test arasında bir seçim yapıldığında pek çok organizasyon anlaşılır biçimde özellikleri seçer. Önlenebilir bir hata kaçınılmaz olarak gerçekleştiğinde hatayı yapan kişiyi suçlamak mantıklı değildir; sorun organizasyonun öncelikleridir.

### Blameless Postmortem Kültürü

Giderek daha fazla organizasyon **blameless postmortem** (suçlamadan olay sonrası analiz) kültürünü benimsemektedir: bir olayın ardından ilgili kişiler, benzeri sorunların nasıl önleneceğini diğerlerinin öğrenmesine olanak tanıdığı için ceza korkusu olmadan yaşananları tam ayrıntısıyla paylaşmaya teşvik edilir.

Bu süreç, iş önceliklerini değiştirme, ihmal edilen alanlara yatırım yapma, ilgili kişilerin teşviklerini değiştirme veya başka bir sistemsel sorunu yönetimin dikkatine sunma ihtiyacını ortaya çıkarabilir.

### Genel Prensip

Bir olay araştırılırken basit cevaplara şüpheyle yaklaşılmalıdır. "Bob o değişikliği deploy ederken daha dikkatli olmalıydı" üretken değildir; ancak "Backend'i Haskell'de yeniden yazmalıyız" da değildir. Bunun yerine yönetim, sosyoteknik sistemin her gün onunla çalışan insanların bakış açısından nasıl işlediğinin ayrıntılarını öğrenme ve bu geri bildirime dayalı iyileştirme adımları atma fırsatını değerlendirmelidir.

### Güvenilirlik Ne Kadar Önemlidir?

Güvenilirlik yalnızca nükleer santraller ve hava trafik kontrolü için değildir; daha sıradan uygulamaların da güvenilir çalışması beklenir.

- İş uygulamalarındaki bug'lar üretkenlik kaybına (rakamların yanlış raporlanması halinde yasal risklere) yol açar
- E-ticaret sitelerinin kesintileri gelir kaybı ve itibar zararı açısından büyük maliyetlere yol açabilir
- Fotoğraf uygulamanızda çocuklarının tüm fotoğraf ve videolarını saklayan bir ebeveyn düşünün; o database aniden bozulursa ne olur?

### Post Office Horizon Skandalı

Güvenilmez yazılımın insanlara nasıl zarar verdiğinin somut bir örneği: 1999–2019 yılları arasında İngiltere'deki Post Office şubelerini yöneten yüzlerce kişi, **muhasebe yazılımının hesaplarında açık göstermesi** nedeniyle hırsızlık veya dolandırıcılıktan mahkûm edildi.

Sonunda bu açıkların büyük bölümünün yazılımdaki bug'lardan kaynaklandığı anlaşıldı ve mahkûmiyetlerin çoğu bozuldu. Bu duruma yol açan şey, İngiliz hukukunun bilgisayarların aksi kanıtlanmadıkça doğru çalıştığını (dolayısıyla bilgisayarlarca üretilen kanıtların güvenilir olduğunu) varsayan bir kuralıdır.

Muhtemelen İngiliz tarihinin en büyük adalet yanlışlığı olan bu durum, yazılımın hatasız olabileceği fikrine yazılım mühendisleri gülebilir; ancak bu, haksız yere hapsedilen, iflas eden ya da intihar eden kişiler için küçük bir teselli olmaktadır.

Bazı durumlarda güvenilirliği feda ederek geliştirme maliyetini azaltmayı seçebiliriz — ancak ne zaman köşe kestiğimizin ve olası sonuçların son derece bilincinde olmalıyız.

---
## 📈 Scalability
### (Ölçeklenebilirlik)

Bir sistem bugün güvenilir biçimde çalışıyor olsa bile bu, gelecekte de güvenilir çalışacağı anlamına gelmez. Bozulmaya yol açan yaygın bir neden, artan yüktür. Belki sistem 10.000 eşzamanlı kullanıcıdan 100.000'e büyümüştür ya da öncekinden çok daha büyük veri hacimlerini işlemektedir.

**Scalability**, bir sistemin artan yükle başa çıkma becerisini tanımlamak için kullandığımız terimdir.

### Scalability Ne Zaman Önemlidir?

> *"Sen Google veya Amazon değilsin. Ölçek konusunda endişelenmekten vazgeç, sadece relational database kullan."*

Bu özdeyişin sizin için geçerli olup olmadığı inşa ettiğiniz uygulama türüne bağlıdır.

Yalnızca az sayıda kullanıcısı olan yeni bir ürün inşa ediyorsanız (örn. bir startup'ta), genel mühendislik hedefi genellikle sistemi mümkün olduğunca basit ve esnek tutmaktır; böylece müşterilerin ihtiyaçları hakkında daha fazla şey öğrendikçe ürününüzün özelliklerini kolayca değiştirebilir ve uyarlayabilirsiniz. Böyle bir ortamda hipotetik gelecek ölçeği için endişelenmek üretken değildir.

**Scalability tek boyutlu bir etiket değildir.** "X ölçeklenebilir" veya "Y ölçeklenmez" demek anlamsızdır. Bunun yerine scalability şu soruları ele alır:
- Sistem belirli bir şekilde büyüyorsa, büyümeyle başa çıkmak için seçeneklerimiz nelerdir?
- Ek yükü kaldırmak için computing kaynaklarını nasıl ekleyebiliriz?
- Mevcut büyüme projeksiyonlarına göre mevcut mimarimizin sınırlarına ne zaman ulaşacağız?

---
### 📊 Yükü Anlamak: Understanding Load

Önce sistemdeki mevcut yükü net bir şekilde anlamanız gerekir. Ancak o zaman büyüme sorularını tartışabilirsiniz: "Yükümüz iki katına çıkarsa ne olur?"

Bu çoğunlukla bir **throughput** ölçümüdür — örneğin:
- Bir servise saniyedeki istek sayısı
- Günlük gelen yeni verinin gigabyte cinsinden miktarı
- Saatteki alışveriş sepeti ödemesi sayısı
- Sosyal ağ vaka çalışmamızdaki eşzamanlı çevrimiçi kullanıcı sayısının zirvesi

Yükün diğer istatistiksel özellikleri de erişim desenlerini ve dolayısıyla scalability gereksinimlerini etkiler:
- Bir database'deki okuma/yazma oranı
- Bir cache'deki hit oranı
- Kullanıcı başına veri öğesi sayısı (vaka çalışmamızdaki takipçiler gibi)
- Ortalama durum mu önemli, yoksa az sayıdaki aşırı durum mu bottleneck oluşturuyor?

### Yük Artışını İki Şekilde Analiz Etmek

Sisteminizi anlayınca yük arttığında ne olduğunu araştırabilirsiniz:

1. Yükü belirli bir şekilde artırdığınızda ve sistem kaynaklarını (CPU, bellek, network bant genişliği vb.) değiştirmediğinizde, sisteminizin performansı nasıl etkilenir?
2. Yükü belirli bir şekilde artırdığınızda, performansı aynı tutmak istiyorsanız kaynakları ne kadar artırmanız gerekir?

Hedef genellikle sistemin performansını **SLA** gereksinimlerinde tutarken sistemin çalıştırma maliyetini de minimumda tutmaktır.

### Linear Scalability

Kaynakları iki katına çıkarmak iki kat yükü aynı performansla kaldırabilmesini sağlıyorsa, **linear scalability** (doğrusal ölçeklenebilirlik) söz konusudur — bu iyi bir şeydir.

Zaman zaman iki kat yükü iki kattan az kaynak artışıyla kaldırmak mümkün olabilir (ölçek ekonomileri veya daha iyi zirve yük dağılımı sayesinde). Çok daha muhtemel olan ise **maliyetin doğrusal olmayan şekilde büyümesidir**. Bunun pek çok nedeni olabilir; örneğin çok fazla veriniz varsa, tek bir yazma isteğini işlemek az veri olduğu durumdan daha fazla iş gerektirebilir.

---
### 🏗️ Mimari Seçenekler: Shared-Memory, Shared-Disk, Shared-Nothing
### Shared-Memory, Shared-Disk, and Shared-Nothing Architectures

Bir servisin donanım kaynaklarını artırmanın en basit yolu, onu daha güçlü bir makineye taşımaktır. Bu yaklaşıma **vertical scaling** veya **scaling up** denir.

### 1. Shared-Memory Architecture (Paylaşımlı Bellek Mimarisi)

Tek bir makinede çoklu süreç veya thread kullanarak paralellik elde edebilirsiniz. Aynı sürece ait tüm thread'ler aynı RAM'a erişebildiğinden bu yaklaşıma **shared-memory architecture** denir.

**Sorun:** Maliyet doğrusal olmayan şekilde büyür. Daha düşük özelliklerin iki katı donanım kaynağına sahip üst düzey bir makine genellikle iki kattan çok daha pahalıdır. Ve bottleneck'ler nedeniyle o makinenin iki kat yükü gerçekten kaldırabilmesi pek olası değildir.

### 2. Shared-Disk Architecture (Paylaşımlı Disk Mimarisi)

Bağımsız CPU'lara ve RAM'lere sahip birden fazla makine kullanır, ancak hızlı bir ağ ile birbirine bağlı makineler arasında paylaşılan bir disk dizisinde veri saklar:
- **NAS (Network-Attached Storage)**
- **SAN (Storage Area Network)**

Bu mimari geleneksel olarak on-premises data warehousing iş yükleri için kullanılmıştır. Ancak **contention** (rekabet) ve **locking**'in (kilitleme) ek yükü shared-disk yaklaşımının ölçeklenebilirliğini sınırlar.

### 3. Shared-Nothing Architecture (Paylaşımsız Mimari)

**Horizontal scaling** veya **scaling out** olarak da bilinir. Her birinin kendi CPU'larına, RAM'ine ve disklerine sahip olduğu birden fazla node'dan oluşan bir distributed system içerir. Node'lar arasındaki koordinasyon geleneksel bir ağ üzerinden yazılım düzeyinde yapılır.

**Avantajları:**
- Doğrusal ölçeklenme potansiyeli
- En iyi fiyat/performans oranı sunan herhangi bir donanımı kullanabilme (özellikle cloud'da)
- Yük arttıkça veya azaldıkça donanım kaynaklarını daha kolay ayarlayabilme
- Sistemi birden fazla datacenter ve region'a dağıtarak daha fazla fault toleransı

**Dezavantajları:**
- Açık **sharding** gerektirir
- Tüm distributed system karmaşıklığına neden olur

### Cloud Native Hibrit Model

Bazı cloud native database sistemleri, depolama ve işlem yürütme için ayrı servisler kullanır: birden fazla compute node aynı depolama servisine paylaşımlı erişimle. Bu model shared-disk mimarisine benzer, ancak eski sistemlerin ölçeklenebilirlik sorunlarından kaçınır. Bir filesystem (NAS) veya block device (SAN) soyutlaması sunmak yerine, depolama servisi database'in özel ihtiyaçları için tasarlanmış özel bir API sunar.

---
### 🧭 Scalability İlkeleri
### Principles for Scalability

Büyük ölçekte çalışan sistemlerin mimarisi genellikle uygulamaya özgüdür. **Magic scaling sauce** (sihirli ölçekleme sosu) diye de bilinen genel, her şeye uyan bir ölçeklenebilir mimari yoktur.

Örneğin her biri 1 kB boyutunda saniyede 100.000 isteği karşılamak için tasarlanmış bir sistem, her biri 2 GB boyutunda dakikada 3 isteğe hizmet etmek için tasarlanmış bir sistemden çok farklı görünür — her iki sistemin aynı veri throughput'una sahip olmasına rağmen (100 MB/saniye).

### Temel İlkeler

1. **Her büyüklük derecesinde mimariyi yeniden düşünün**: Hızlı büyüyen bir serviste çalışıyorsanız, her büyüklük derecesi yük artışında mimariyi yeniden düşünmeniz büyük olasılıkla gerekecektir. Uygulamanın ihtiyaçları muhtemelen evrim geçireceğinden, gelecekteki ölçekleme ihtiyaçlarını birden fazla büyüklük derecesi önceden planlamak genellikle değmez.

2. **Küçük, bağımsız bileşenler**: Scalability için iyi bir genel prensip, sistemi büyük ölçüde bağımsız çalışabilen daha küçük bileşenlere bölmektir. Bu, microservices, sharding, stream processing ve shared-nothing mimarilerinin temel ilkesidir.

3. **Gereksiz karmaşıklıktan kaçının**: Başka iyi bir prensip, şeyler gerekenden daha karmaşık yapmamaktır. Tek makineli bir database işi görecekse, karmaşık bir distributed kurulumdan muhtemelen daha iyidir. **Autoscaling sistemleri** (talebe yanıt olarak kaynakları otomatik ekleyen veya kaldıran) harika görünür; ancak yükünüz oldukça öngörülebilirse, manuel ölçeklenen bir sistem daha az operasyonel sürprizle karşılaşabilir. 5 servise sahip bir sistem, 50 servise sahip olandan daha basittir. İyi mimariler genellikle yaklaşımların pragmatik bir karışımını içerir.

---
## 🔨 Maintainability
### (Bakım Kolaylığı)

Yazılım aşınmaz veya malzeme yorgunluğuna uğramaz, bu nedenle mekanik nesneler gibi bozulmaz. Ancak bir uygulamanın gereksinimleri sıkça değişir, yazılımın çalıştığı ortam değişir (bağımlılıkları ve altta yatan platform gibi) ve düzeltilmesi gereken bug'ları olabilir.

Yazılım maliyetinin büyük bölümünün **başlangıç geliştirmesinde değil, süregelen bakımında** olduğu yaygın biçimde kabul görür: bug'ları düzeltme, sistemleri çalışır tutma, arızaları araştırma, yeni platformlara uyarlama, yeni kullanım senaryoları için değiştirme, **technical debt** ödeme ve yeni özellikler ekleme.

### Legacy Sistemler

Bakım özellikle **legacy sistemler** için karmaşık olabilir:
- Uzun süredir başarıyla çalışan bir sistem, bugün çok az mühendisinin anladığı güncel olmayan teknolojiler (mainframe ve COBOL kodu gibi) kullanıyor olabilir
- İnsanlar organizasyondan ayrıldıkça sistemin belirli bir şekilde tasarlanmasının nasıl ve neden olduğuna dair kurumsal bilgi kaybolmuş olabilir
- Başkalarının hatalarını düzeltmek de gerekebilir

Bugün oluşturduğumuz her sistem, yeterince uzun hayatta kalacak kadar değerliyse bir gün **legacy sistem** haline gelecektir. Yazılımımızı bakımla göz önünde bulundurarak tasarlamalıyız.

### Üç Temel Maintainability İlkesi

| İlke | Açıklama |
|---|---|
| **Operability** | Organizasyonun sistemi sorunsuz çalıştırmasını kolaylaştırın |
| **Simplicity** | İyi anlaşılmış, tutarlı kalıplar ve yapılar kullanarak ve gereksiz karmaşıklıktan kaçınarak yeni mühendislerin sistemi anlamasını kolaylaştırın |
| **Evolvability** | Mühendislerin gelecekte sistemi değiştirmesini ve gereksinimler değiştikçe onu uyarlamasını ve genişletmesini kolaylaştırın |

---
### ⚙️ Operability: Operasyonlar İçin Hayatı Kolaylaştırmak
### Operability: Making Life Easy for Operations

"İyi operasyonlar kötü (veya eksik) yazılımın sınırlamalarının üstesinden genellikle gelebilir; ancak iyi yazılım kötü operasyonlarla güvenilir biçimde çalışamaz."

Binlerce makineden oluşan büyük ölçekli sistemlerde manuel bakım makul olmayan bir maliyete sahip olur; otomasyon zorunludur. Ancak otomasyon çift kenarlı bir kılıç olabilir:
- Otomatik olarak ele alınamayan durumlar (nadir arıza senaryoları gibi) her zaman olacaktır ve otomatik olarak ele alınamayan durumlar en karmaşık olanlar olduğundan, daha fazla otomasyon bunları çözebilecek daha yetenekli bir operasyon ekibi gerektirir.
- Yanlış giden otomatik bir sistemi sorun gidermek, operatörün bazı eylemleri manuel olarak gerçekleştirmesine dayanan bir sistemden genellikle daha güçtür.

Bu nedenle daha fazla otomasyon her zaman operabilite için daha iyi değildir. Ancak belirli miktarda otomasyon önemlidir — optimal nokta uygulamanızın ve organizasyonunuzun özelliklerine bağlıdır.

### İyi Operability Nedir?

İyi operability, rutin görevleri kolaylaştırmak ve operasyon ekibinin yüksek değerli faaliyetlere odaklanmasını sağlamak anlamına gelir. Veri sistemleri şunları yaparak yardımcı olabilir:

- Sistemin temel metriklerini kontrol etmek için izleme araçlarına izin verme ve sistemin çalışma zamanı davranışına içgörü sağlamak için observability araçlarını destekleme
- Bireysel makinelere bağımlılıktan kaçınma (sistem bütününde kesintisiz çalışmaya devam ederken makinelerin bakım için devre dışı bırakılabilmesi)
- İyi dokümantasyon ve anlaşılması kolay bir operasyonel model sağlama ("X yaparsam Y olur")
- İyi varsayılan davranış sağlama, ancak gerektiğinde yöneticilere varsayılanları geçersiz kılma özgürlüğü tanıma
- Uygun durumlarda kendi kendini iyileştirme; ama gerektiğinde yöneticilere sistem durumu üzerinde manuel kontrol de verme
- Öngörülebilir davranış sergileme ve sürprizleri en aza indirme

---
### 🧩 Simplicity: Karmaşıklığı Yönetmek
### Simplicity: Managing Complexity

Küçük yazılım projeleri çok basit ve ifade gücü yüksek koda sahip olabilir; ancak projeler büyüdükçe genellikle çok karmaşık ve anlaması güç hale gelir. Bu karmaşıklık, sistemde çalışması gereken herkesi yavaşlatır ve bakım maliyetini daha da artırır. Karmaşıklığa gömülü bir yazılım projesine bazen **big ball of mud** (büyük bir çamur topu) denir.

Karmaşıklık bakımı zorlaştırdığında bütçeler ve zamanlamalar çoğunlukla aşılır. Karmaşık yazılımda, değişiklik yaparken bug ortaya çıkma riski de daha yüksektir. Sistem, geliştiricilerin anlaması ve akıl yürütmesi için ne kadar zorsa, gizli varsayımlar, istenmeyen sonuçlar ve beklenmedik etkileşimler o kadar kolay gözden kaçırılır.

Tersine, karmaşıklığı azaltmak yazılımın bakım kolaylığını büyük ölçüde iyileştirir; bu nedenle sadelik, inşa ettiğimiz sistemlerin temel hedefi olmalıdır.

### Essential vs Accidental Complexity

Karmaşıklık hakkında akıl yürütmeye yönelik bir girişim, bunu iki kategoriye ayırır (Fred Brooks'un "No Silver Bullet" makalesi):

- **Essential complexity** (özsel karmaşıklık): Uygulamanın problem alanında içkin olan karmaşıklık
- **Accidental complexity** (tesadüfi karmaşıklık): Yalnızca araçlarımızın sınırlamaları nedeniyle ortaya çıkan karmaşıklık

Ne yazık ki bu ayrım da kusurludur; çünkü özsel ve tesadüfi arasındaki sınırlar araçlarımız geliştikçe kayar.

### Abstraction: Karmaşıklığı Yönetmenin En İyi Aracı

Karmaşıklığı yönetmek için sahip olduğumuz en iyi araçlardan biri **abstraction** (soyutlama)dır. İyi bir abstraction, büyük miktarda uygulama ayrıntısını temiz, anlaşılması kolay bir **façade**'ın (cephe) arkasına gizleyebilir. İyi bir abstraction aynı zamanda çok çeşitli uygulamalar için kullanılabilir.

Bu yeniden kullanım, benzer bir şeyi birden fazla kez yeniden implement etmekten daha verimli olmakla kalmaz; aynı zamanda soyutlanan bileşendeki kalite iyileştirmeleri onu kullanan tüm uygulamalara fayda sağladığından daha yüksek kaliteli yazılıma yol açar.

**Abstraction örnekleri:**
- **Yüksek seviyeli programlama dilleri**: Makine kodu, CPU register'ları ve sistem çağrılarını gizleyen abstraction'lar
- **SQL**: Karmaşık disk ve bellek içi veri yapılarını, diğer client'lardan gelen eşzamanlı istekleri ve çökmelerden sonraki tutarsızlıkları gizleyen abstraction

Uygulama kodu için karmaşıklığını azaltmaya yönelik abstraction'lar şu metodolojiler kullanılarak oluşturulabilir:
- **Design patterns** (tasarım kalıpları)
- **Domain-Driven Design (DDD)**

Bu kitap bu tür uygulamaya özgü abstraction'larla değil, üzerine uygulamalarınızı inşa edebileceğiniz genel amaçlı abstraction'larla ilgilenir: database transaction'ları, index'ler ve event log'lar gibi.

---
### 🔄 Evolvability: Değişimi Kolaylaştırmak
### Evolvability: Making Change Easy

Sisteminizin gereksinimlerinin sonsuza dek değişmeyeceği son derece düşük bir ihtimaldir. Büyük olasılıkla sürekli değişim halinde olacaklardır:
- Yeni gerçekler öğrenirsiniz
- Önceden öngörülmemiş kullanım senaryoları ortaya çıkar
- İş öncelikleri değişir
- Kullanıcılar yeni özellikler talep eder
- Yeni platformlar eski platformların yerini alır
- Yasal veya düzenleyici gereksinimler değişir
- Sistemin büyümesi mimari değişiklikleri zorunlu kılar

### Agile ve Evolvability

Organizasyonel süreçler açısından **Agile** çalışma modelleri, değişime uyum sağlamak için bir çerçeve sunar. Agile topluluğu aynı zamanda sıkça değişen ortamlarda yazılım inşa ederken faydalı teknik araçlar ve süreçler geliştirmiştir:
- **TDD (Test-Driven Development)** — Teste dayalı geliştirme
- **Refactoring** — Kodu yeniden düzenleme

Bu kitapta, farklı özelliklere sahip birkaç uygulama veya servisten oluşan bir sistem düzeyinde **evolvability**'yi artırmanın yolları aranır.

### Irreversibility: Değişimi Zorlaştıran Temel Faktör

Büyük sistemlerde değişimi zorlaştıran önemli bir faktör **irreversibility** (geri döndürülemezliktir).

Örneğin, bir veritabanından diğerine geçiş yapıyorsunuz. Yeni sistemde sorunlar olması durumunda eski sisteme geri dönemiyorsanız, kolayca geri dönebileceğiniz duruma kıyasla riskler çok daha yüksektir. Bu nedenle geri döndürülemez eylemler çok dikkatli alınmalıdır. **Irreversibility'yi minimize etmek esnekliği artırır.**

Bir veri sistemini değiştirme ve değişen gereksinimlere uyarlama kolaylığı, sistemin basitliği ve abstraction'larıyla yakından bağlantılıdır. Gevşek bağlı (loosely coupled), basit sistemler genellikle sıkı bağlı (tightly coupled), karmaşık olanlardan daha kolay değiştirilir. Bu son derece önemli bir fikir olduğundan, veri sistemi düzeyinde çevikliği ifade etmek için farklı bir kelime kullanılır: **evolvability**.

---
## 📋 Bölüm Özeti

Bu bölümde birkaç **nonfunctional requirement** örneği incelendi: performance, reliability, scalability ve maintainability. Bu konular aracılığıyla kitabın geri kalanında da geçerli olacak ilkeler ve terminolojiyle tanışıldı.

### Ana Çıkarımlar:

1. **Performance**: Response time'ı tek bir sayı yerine percentile dağılımı olarak düşünmek gerekir. **p50**, **p95**, **p99**, **p999** gibi percentile'lar kullanıcı deneyimini farklı boyutlarda ölçer. **SLO** ve **SLA**'lar bu metriklere dayanır. Sistemi aşırı yüklüyorsa **metastable failure**'ı önlemek için exponential backoff, circuit breaker ve load shedding gibi teknikler kullanılır.

2. **Reliability**: **Fault tolerance** teknikleri, bir bileşen arızalı olsa bile sistemin hizmet sağlamaya devam etmesini mümkün kılar. **Hardware faults** çoğunlukla bağımsız olduğundan redundancy ile tolere edilebilir; **software faults** yüksek korelasyonlu olduğundan daha zordur. İnsan hataları, blameless postmortem kültürü ve sistemsel bir bakış açısıyla ele alınmalıdır.

3. **Scalability**: Tek boyutlu bir etiket değildir; her sistemin özgün ölçekleme ihtiyaçları vardır. **Shared-memory**, **shared-disk** ve **shared-nothing** mimarileri farklı trade-off'lar sunar. Genel ilke: sistemi bağımsız bileşenlere bölmek ve gereksiz karmaşıklıktan kaçınmak.

4. **Maintainability**: **Operability** (çalıştırması kolay), **Simplicity** (anlaması kolay) ve **Evolvability** (değiştirmesi kolay) olmak üzere üç temel boyutu vardır. Abstraction karmaşıklığı yönetmenin en güçlü aracıdır.

---

### 🔑 Temel Terimler Sözlüğü

| Terim | Türkçe Karşılığı / Açıklama |
|---|---|
| **Functional requirements** | İşlevsel gereksinimler: Uygulamanın ne yapması gerektiği |
| **Nonfunctional requirements** | İşlevsel olmayan gereksinimler: Hız, güvenilirlik, güvenlik, bakım kolaylığı |
| **Response time** | Yanıt süresi: İstekten yanıta kadar geçen süre |
| **Service time** | Servis süresi: Servisin aktif işleme yaptığı süre |
| **Throughput** | Verim: Saniyedeki istek sayısı veya veri hacmi |
| **Latency** | Gecikme: İsteğin aktif işlenmediği süre |
| **Queueing delay** | Kuyruk gecikmesi: CPU müsait olana kadar bekleme süresi |
| **Head-of-line blocking** | Yavaş bir isteğin arkadasındakileri tıkaması |
| **Percentile (p50, p95, p99)** | Yüzdelik dilim: Response time dağılımını ölçer |
| **Tail latency** | Kuyruk gecikmesi: Yüksek percentile'lardaki yavaş yanıtlar |
| **Tail latency amplification** | Birden fazla backend çağrısı gerektiren isteklerde gecikmeni büyümesi |
| **SLO** | Service Level Objective: Dahili performans hedefi |
| **SLA** | Service Level Agreement: SLO ihlalinin sonuçlarını belirleyen sözleşme |
| **Metastable failure** | Sistemi aşırı yükte tutan kısır döngü |
| **Retry storm** | Sistemi daha da aşırı yükleyen kaskad yeniden deneme fırtınası |
| **Exponential backoff** | Yeniden denemeler arasındaki süreyi katlanarak artırma |
| **Circuit breaker** | Hata veren servise geçici olarak istek göndermeyi durdurma |
| **Load shedding** | Aşırı yük altında istekleri proaktif olarak reddetme |
| **Backpressure** | İstemcilere yavaşlamalarını söyleme mekanizması |
| **Fault** | Arıza: Sistemin bir parçasının çalışmayı durdurması |
| **Failure** | Başarısızlık: Sistemin bir bütün olarak hizmet vermeyi durdurması |
| **SPOF (Single Point of Failure)** | Tek başına tüm sistemi çökertebilen kritik nokta |
| **Fault tolerance** | Belirli arızalara rağmen hizmet vermeye devam etme |
| **Fault injection** | Kasıtlı hata oluşturarak tolerans mekanizmasını test etme |
| **Chaos engineering** | Kasıtlı hata enjekte ederek fault tolerance'a güven inşa etme |
| **RAID** | Redundant Array of Independent Disks: Çoklu disk yedekleme |
| **Availability zone** | Fiziksel olarak ayrı datacenter bölgesi |
| **Rolling upgrade** | Hizmeti durdurmadan kademeli güncelleme |
| **ECC (Error-Correcting Code)** | Hata düzelten bellek |
| **Blameless postmortem** | Suçlamadan olay sonrası analiz |
| **Scalability** | Ölçeklenebilirlik: Artan yükle başa çıkma becerisi |
| **Linear scalability** | Kaynaklar iki katına çıkınca yük kapasitesi de iki katına çıkar |
| **Vertical scaling / Scaling up** | Daha güçlü bir makineye geçiş |
| **Horizontal scaling / Scaling out** | Daha fazla makine ekleme |
| **Shared-memory architecture** | Tüm thread'lerin aynı RAM'i paylaştığı mimari |
| **Shared-disk architecture** | Makinelerin ortak disk üzerinden veri paylaştığı mimari |
| **Shared-nothing architecture** | Her node'un kendi CPU/RAM/diskine sahip olduğu dağıtık mimari |
| **NAS** | Network-Attached Storage: Ağa bağlı depolama |
| **SAN** | Storage Area Network: Depolama alan ağı |
| **Sharding** | Veriyi birden fazla node'a bölme |
| **Magic scaling sauce** | Herkese uyan sihirli ölçekleme çözümü — var olmaz |
| **Maintainability** | Bakım kolaylığı: Uzun vadede sistemin kolay idame edilmesi |
| **Technical debt** | Teknik borç: Kısa vadeli kararların uzun vadede yarattığı yük |
| **Legacy system** | Eski teknoloji kullanan miras sistem |
| **Operability** | Sistemin çalıştırılması kolay olması |
| **Simplicity** | Sistemin anlaşılması kolay olması |
| **Evolvability** | Sistemin değiştirilmesi kolay olması |
| **Abstraction** | Karmaşık detayları temiz bir façade arkasına gizleme |
| **Essential complexity** | Problemin doğasından gelen kaçınılmaz karmaşıklık |
| **Accidental complexity** | Araçların sınırlamalarından kaynaklanan gereksiz karmaşıklık |
| **Big ball of mud** | Karmaşıklığa gömülü, anlaşılmaz yazılım |
| **Domain-Driven Design (DDD)** | Karmaşıklığı yönetmek için domain odaklı tasarım metodolojisi |
| **TDD** | Test-Driven Development: Teste dayalı geliştirme |
| **Irreversibility** | Geri döndürülemezlik: Evolvability'yi kısıtlayan faktör |
| **Fan-out** | Bir isteğin birden fazla downstream isteğe yol açması |
| **Materialization** | Sorgu sonuçlarını önceden hesaplama ve saklama |
| **Materialized view** | Önceden hesaplanmış sorgu sonucu cache |
| **Polling** | Client'ın periyodik olarak sunucuyu yoklaması |
| **HdrHistogram / t-digest / DDSketch** | Verimli percentile hesaplama algoritmaları |